# <strong>TCP SYN flood</strong>

This exercise demonstrates a well-known denial-of-service attack, called <strong>TCP SYN flood</strong>. Students will be able to create a real attack using SPHERE tools, and to observe its effect on legitimate traffic. Afterwards, they will be asked to apply a known defense against SYN flood known as <strong>SYN cookies</strong>, repeat the attack and observe the protection.

This exercise helps students learn the following concepts: (1) How TCP/IP works and how its design can be misused for attacks, (2) How easy it is to perpetrate a DoS attack, with fully legitimate traffic and at a low rate, (3) How easy it is to protect machines from this type of attacks via built-in OS mechanisms. Additionally, extra credit questions improve a student's understanding of how networks and TCP/IP work. 

<strong>This lab will contain four topics:</strong>

1. Generating Legitimate Traffic
2. Turning Off SYN Cookies
3. Generating Attack Traffic
4. Collecting Statistics

### Step 0: Starting the Lab

Click the button to begin creating the experiment.

<strong>Note:</strong> If your buttons are not displaying, click on the <img width='20px' height='20px' style='margin-left: 1px;' src='resources/fast_forward.png'> icon at the top of your notebook to render all widgets.

In [1]:
# Click the button below to start your lab.
import os
import subprocess
import re
import threading
import queue
import time
import sys
from IPython.display import display, HTML
import ipywidgets as widgets
import logging
import tarfile
from pathlib import Path

# This is a troublesome file that will throw unnecessary warnings and other errors.
# Delete it if it exists. (Wasn't an issue in older Jupyter versions.)
# !(rm -f ~/.local/share/jupyter/nbsignatures.db)

# Global variable for the checker script.
runAllSteps = False

# Adding the "resource/" directory so that we can import the start.py file.
module_dir = os.path.join(os.getcwd(), 'resources')
if module_dir not in sys.path:
    sys.path.append(module_dir)

# Importing the prepare_lab function.
from functions import *

# Defining some stuff for the output below.
output0 = widgets.Output()
startButton = widgets.Button(description="Start Lab")
labname = "synflood"

# Defining the button handler for startButton.
def on_start_clicked(b):
    prepare_lab(labname, output0)

# Providing on_click functionality.
startButton.on_click(on_start_clicked)

# Display button and output area.
display(startButton, output0)

Button(description='Start Lab', style=ButtonStyle())

Output()

<hr>

If you previously stopped your lab, you may restore your progress below by clicking "Load Lab". <u>You do not have to load your lab if you signed out, closed your notebook, or exited your node(s) or XDC by using ```exit```.</u>

In [2]:
# Click the button below to load your lab.
def loadlab(b):
    load_lab(labname, output0_2)

# Creating the button.
loadButton = widgets.Button(description="Load Lab")

# Creating an output area.
output0_2 = widgets.Output()

# Run the command on click.
loadButton.on_click(loadlab)

# Display the output.
display(loadButton, output0_2)

Button(description='Load Lab', style=ButtonStyle())

Output()

## <strong>Introduction</strong>

All students should have completed an introductory networking course with grade B or better.

- <a href="http://en.wikipedia.org/wiki/SYN_flood">Short summary of SYN flood attack on Wikipedia</a>
- SYN flood attacks in the <a href="http://www.amazon.com/Internet-Denial-Service-Mechanisms-Networking/dp/0131475738/ref=sr_1_1?ie=UTF8&s=books&qid=1212642071&sr=8-1">Internet Denial of Service</a> book (optional reading)
- <a href="http://cr.yp.to/syncookies.html">SYN cookie overview</a>
- <a href="http://www.tcpdump.org/tcpdump_man.html">Tcpdump's man page</a>

Denial of service attacks deny service to legitimate clients by tying up resources at the server with a flood of legiitmate-looking service requests or junk traffic. Before proceeding to the assignment instructions make sure that you understand how TCP SYN flood attack works, which resource it ties up and how, and how syncookies help mitigate this attack. 

Upon starting your lab, you will have a topology that looks like this.

<div style="text-align: center; padding-right: 40px"><img src="resources/synflood/topo.png"></div>

### Step 1: Create a Web Traffic Stream

You will start this lab by simulating network traffic. This can be easily done by creating a Bash script and using the `curl` command. If you are unfamiliar with `curl`, you may view the tutorial <a href="https://curl.se/docs/tutorial.html">here</a>.

<strong>Your task:</strong> SSH onto the `client` node by typing `ssh client`. Inside of your home directory (`/home/USERNAME_GOES_HERE`), create a file named `stream.sh`. You will need to include a shebang, which will allow you to call the script from the command line. A sample script will look like this:

```
#!/bin/bash

(Your Solution Here)
```

Inside of `(Your Solution Here)`, create a `curl` call that gets `index.html` from the `server` every second. The `index.html` script can be found in `/var/www/html` inside of `server`.

<strong>Requirements:</strong>
- Use a <a href="https://www.warp.dev/terminus/bash-while-loop">while true</a> loop.
- Use a <a href="https://en.wikipedia.org/wiki/Sleep_(command)">sleep</a> command.
- The script does not create errors.

Once you have a working script, click on the button below to check your work. The notebook will test your script, and ensure that you have `curl` within your script.

In [3]:
# Click the button below to check your work.
step1Complete = False

# Function to check the permissions.
def step_1():
    # Important variables that must be accessed outside of this function.
    global step1Complete, result

    with output1:
        output1.clear_output()
        display(HTML("<span>Testing your script. Please wait.</span><span><img width='14px' height='14px' style='margin-left: 5px;' src='resources/loading.gif'></span>"))
    
    # This subprocess statement is a little different. Need to initiate environment variables at the same time when running the command.
    result = subprocess.run('ssh -i /home/USERNAME_GOES_HERE/.ssh/merge_key USERNAME_GOES_HERE@client /home/.checker/step_1.py', shell=True, capture_output=True, text=True)
    
    if (result.returncode == 0):
        output1.clear_output()
        with output1:
            display(HTML("<span style='color: green;'>Your script has passed all checks.</span>"))
            step1Complete = True

    elif (result.returncode == 1 or result.returncode == 8):
        output1.clear_output()
        with output1:
            display(HTML("<span style='color: red;'>There was an error checking this step. Please contact your professor/TA.</span>"))
            step1Complete = False

    elif (result.returncode == 2):
        output1.clear_output()
        with output1:
            display(HTML("<span style='color: red;'>The <code>stream.sh</code> file cannot be found in your home directory. Ensure that it's in the home directory on your <code>client</code> node.</span>"))
            step1Complete = False
    
    elif (result.returncode == 3):
        output1.clear_output()
        with output1:
            display(HTML("<span style='color: red;'>You are not using <code>curl</code> in your command. Please include it in your script.</span>"))
            step1Complete = False

    elif (result.returncode == 4):
        output1.clear_output()
        with output1:
            display(HTML("<span style='color: red;'>index.html is not found in your script, and this file must be called with curl. Make sure to call the file (/var/www/html/index.html) from the server node. Please include it in your script.</span>"))
            step1Complete = False

    elif (result.returncode == 5):
        output1.clear_output()
        with output1:
            display(HTML("<span style='color: red;'>You are not using \"sleep\" in your command. Please include it in your script.</span>"))
            step1Complete = False

    elif (result.returncode == 6):
        output1.clear_output()
        with output1:
            display(HTML("<span style='color: red;'>You are not using a \"while true\" loop in your command. Please include it in your script.</span>"))
            step1Complete = False

    elif (result.returncode == 7):
        output1.clear_output()
        with output1:
            display(HTML("<span style='color: red;'>Please add executable permissions for your script.</span>"))
            step1Complete = False

    elif (result.returncode == 9):
        output1.clear_output()
        with output1:
            display(HTML("<span style='color: red;'>Your script ran with errors. Please check your work and try again.</span>"))
            step1Complete = False 

def check_step_1(b):
    if (warn_student(labname)):
        output1.clear_output()
        with output1:
            display(HTML("<span style='color: red;'><strong>WARNING:</strong> You have an autosaved lab that you have not yet loaded. If you would like to load your progress, click \"Load Lab\" at the top of the notebook. Otherwise, clicking on this button again will assume you're restarting the lab!</span>"))
    else:
        step_1()

        # Auto-save.
        if (not runAllSteps):
            trigger_save(labname, "1", result.returncode)

# Creating the button.
button = widgets.Button(description="Check File")

# Creating an output area.
output1 = widgets.Output()

# Run the command on click.
button.on_click(check_step_1)

# Display the output.
display(button, output1)

Button(description='Check File', style=ButtonStyle())

Output()

### Step 2: Disabling SYN Cookies

In this second question, you are going to experiment how to tinker with SYN cookies in your Linux environment.

SYN cookies are often on by default in Linux and FreeBSD. To check if they are on, type the following on `server` node: `sudo sysctl net.ipv4.tcp_syncookies`

You should see `1` as the result. 

<strong>Your task:</strong> SYN cookies must be set to zero for the demo to work. Type the following two commands on the `server` node:
```
sudo sysctl -w net.ipv4.tcp_syncookies=0
sudo sysctl -w net.ipv4.tcp_max_syn_backlog=10000
```

Verify that SYN cookies are turned off, then click on the button below to check your work.

In [4]:
# Click the button below to check your work.
def step_2():
    # Important variables that must be accessed outside of this function.
    global step2Complete, result

    with output2:
        output2.clear_output()
        display(HTML("<span><img width='12px' height='12px' style='margin-left: 3px;' src='resources/loading.gif'></span>"))
    
    result = subprocess.run(['ssh', '-i', '/home/USERNAME_GOES_HERE/.ssh/merge_key', 'USERNAME_GOES_HERE@server', '/home/.checker/step_2.py'])

    if (result.returncode == 0):
        output2.clear_output()
        with output2:
            display(HTML("<span style='color: green;'>You have turn off the syncookies. Please continue to the next step.</span>"))
            step2Complete = True
            
    elif (result.returncode == 1):
        output2.clear_output()
        with output2:
            display(HTML("<span style='color: red;'>You have not turned off syncookies yet. Please follow the steps above to disable them.</span>"))
            step2Complete = False
            
    elif (result.returncode == 2):
        output2.clear_output()
        with output2:
            display(HTML("<span style='color: red;'>There was an error checking your work. Please contact your professor/TA.</span>"))
            step2Complete = False

def check_step_2(b):
    if (warn_student(labname)):
        output2.clear_output()
        with output2:
            display(HTML("<span style='color: red;'><strong>WARNING:</strong> You have an autosaved lab that you have not yet loaded. If you would like to load your progress, click \"Load Lab\" at the top of the notebook. Otherwise, clicking on this button again will assume you're restarting the lab!</span>"))
    else:
        step_2()

        # Auto-save.
        if (not runAllSteps):
            trigger_save(labname, "2", result.returncode)

# Creating the button.
button = widgets.Button(description="Check File")

# Creating an output area.
output2 = widgets.Output()

# Run the command on click.
button.on_click(check_step_2)

# Display the output.
display(button, output2)

Button(description='Check File', style=ButtonStyle())

Output()

### Step 3: The Flooder Tool

Now, you will create your first SYN flood attack by using the Flooder tool.

<strong>Your task:</strong> Create a SYN flood between the `attacker` and the `server` nodes, using the Flooder tool. You can type `flooder` on the `attacker` node's command line to get a man page for the tool. For your command, send 100 packets per second, and send through IP protocol 6.

For example: `flooder --dst server --src 1.2.0.0 --srcmask 255.255.0.0 --highrate 100 --proto 6` will send a flood of 100 SYN packets per second to the target called `server`, spoofing addresses from 1.2.0.0/16 range. You should make sure to spoof within <strong>1.1.2.0</strong> range (use mask <strong>255.255.255.0</strong>). 

Most flooder commands require a "sudo" in front. 

Once you have a functional command, type it into the text entry below. Your command will be tested, and ensure that it creates a SYN flood.

In [5]:
# Click the button below to check your work.
step3Complete = False

# Function to check if the student's answer was correct.
def step_3():
    global step3Complete, result
    with output3:
        output3.clear_output()
        display(HTML("<span><img width='14px' height='14px' style='margin-left: 3px;' src='resources/loading.gif'></span>"))

    if userInput3.value == "":
        output3.clear_output()
        with output3:
            display(HTML("<span style='color: red;'>You did not provide input for this step.</span>"))
            step3Complete = False
    else:
        escaped_user_input = re.sub(r"(\"|\')", r"'\''", userInput3.value.strip())
        test_command = [
            "ssh", "-i", "/home/USERNAME_GOES_HERE/.ssh/merge_key",
            "USERNAME_GOES_HERE@attacker",
            f"/home/.checker/step_3.py", f"\"{escaped_user_input}\"", ">/dev/null"
        ]
        result = subprocess.run(test_command, text=True, capture_output=True)

        save_command = [
            "ssh", "-i", "/home/USERNAME_GOES_HERE/.ssh/merge_key",
            "USERNAME_GOES_HERE@attacker",
            f"echo '{escaped_user_input}' > /home/.checker/responses/step_3_answer.txt"
        ]
        save_result = subprocess.run(save_command, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
       
        if (result.returncode == 0):
            output3.clear_output()
            with output3:
                display(HTML("<span style='color: green;'>You have the minimum amount of parameters for a valid flooder attack. You may continue to the next step.</span>"))
                step3Complete = True
                
        elif (result.returncode == 1):
            output3.clear_output()
            with output3:
                display(HTML("<span style='color: red;'>There was an error checking your work. Please contact your professor/TA.</span>"))
                step3Complete = False

        elif (result.returncode == 2):
            output3.clear_output()
            with output3:
                display(HTML("<span style='color: red;'>Ensure that your command starts with <code>sudo flooder</code>.</span>"))
                step3Complete = False

        elif (result.returncode == 3):
            output3.clear_output()
            with output3:
                display(HTML("<span style='color: red;'>You are missing some parameters required for your payload, or you have an incorrect value somewhere (like an invalid mask address). If you are certain that your values are correct, check to make sure that you don't have spaces immediately after \"--\" and your parameter name.</span>"))
                step3Complete = False

       
def check_step_3(b):
    if (warn_student(labname)):
        output3.clear_output()
        with output3:
            display(HTML("<span style='color: red;'><strong>WARNING:</strong> You have an autosaved lab that you have not yet loaded. If you would like to load your progress, click \"Load Lab\" at the top of the notebook. Otherwise, clicking on this button again will assume you're restarting the lab!</span>"))
    else:
        step_3()

        # Auto-save.
        if (not runAllSteps):
            trigger_save(labname, "3", result.returncode, userInput3.value)

# Retrieve the student's response. First, create a loading spinner, since this could take a second or two.
loading3 = widgets.Output()
display(loading3)
with loading3:
    loading3.clear_output()
    display(HTML("<span>Loading your saved response... <img width='14px' height='14px' style='margin-left: 3px;' src='resources/loading.gif'></span>"))

# Creating a handler that will automatically convert backticks into single quotes in real time. This is to prevent issues when students use backticks.
def on_input_change(change):
    # Replace backticks with single quotes in the new value.
    new_val = change['new'].replace("`", "'")
    # Only update if there's a change (to avoid infinite recursion).
    if new_val != change['new']:
        change['owner'].value = new_val

# Creating a text area.
userInput3 = widgets.Text(
    placeholder='Type your payload here',
    description='Payload:',
    layout=widgets.Layout(width='90%')
)

userInput3.observe(on_input_change, names='value')

# Checking if the step has been answered.
result = subprocess.run([
    "ssh", "-o", "StrictHostKeyChecking=no", "-i",
    "/home/USERNAME_GOES_HERE/.ssh/merge_key",
    "USERNAME_GOES_HERE@attacker",
    "cat", "/home/.checker/responses/step_3_answer.txt"
], capture_output=True, text=True)

userInput3.value = result.stdout

# After the student's response was loaded, clear the output.
loading3.clear_output()

# Creating the button.
button = widgets.Button(description="Test Payload")

# Creating an output area.
output3 = widgets.Output()

# Run the command on click.
button.on_click(check_step_3)

# Display the output.
display(userInput3, button, output3)

Output()

Text(value='sudo flooder --proto 6 --highrate 100 --dst server --src 1.1.2.0 --srcmask 255.255.255.0\n', descr…

Button(description='Test Payload', style=ButtonStyle())

Output()

### Step 4: Collecting Statistics (Part 1)

You will now collect `tcpdump` statistics on `client` with and without syncookies. Then, you will calculate connection duration and draw graphs of connection duration on y-axis and connection start time on x-axis.

<strong>Before you start the next task:</strong> Stop all traffic by stopping your legitimate `client`'s script and flooder if you haven't done so already. Navigate to the `client` and start a `tcpdump` with the following command:
```
ip route get 5.6.7.8
```

You should see something like this as a result: 
```
5.6.7.8 via 1.1.2.2 dev eth2  src 1.1.2.3
   cache mtu 1500 advmss 1460 metric 10 64
```

Thus the interface name leading to `5.6.7.8` is <strong>eth2</strong>. To see the traffic flowing, type: 
```
sudo tcpdump -nn -i eth2 
```

then generate some traffic and restart your legitimate client code.

<strong>Your task:</strong> Ensure that your SYN cookies are remained off. Then, with the information above, perform the following steps:

- Start legitimate traffic (the script you made in Step 1)
- After 30 seconds, start the attack (the command you constructed in Step 3)
- After another 120 seconds, stop the attack
- After another 30 seconds, stop the legitimate traffic
- Stop the `tcpdump` on the `client`. Inside of your home (`/home/USERNAME_GOES_HERE`) directory on `client`, save the file as `tcpdump_cookies_off.txt`.

Make sure that you start `tcpdump` with the options provided above. 

By clicking "Check File" below, it will check that a file called `/home/USERNAME_GOES_HERE/tcpdump_cookies_off.txt` exists on your `client` node. This step doesn't check if your output is valid, so ensure that you call the `tcpdump` command correctly.

In [6]:
# Click the button below to check your work.
step4Complete = False

# Function to check the permissions.
def step_4():
    # Important variables that must be accessed outside of this function.
    global step4Complete, result

    with output4:
        output4.clear_output()
        display(HTML("<span><img width='14px' height='14px' style='margin-left: 5px;' src='resources/loading.gif'></span>"))
    
    # This subprocess statement is a little different. Need to initiate environment variables at the same time when running the command.
    result = subprocess.run(['ssh', '-i', '/home/USERNAME_GOES_HERE/.ssh/merge_key', 'USERNAME_GOES_HERE@client', '[ -f ~/tcpdump_cookies_off.txt ] && echo 0 || echo 1'], capture_output=True, text=True)

    if (result.stdout == "0\n"):
        output4.clear_output()
        with output4:
            display(HTML("<span style='color: green;'><code>tcpdump_cookies_off.txt</code> has been found, and is saved in your submission</span>"))
            step4Complete = True

    elif (result.stdout == "1\n"):
        output4.clear_output()
        with output4:
            display(HTML("<span style='color: red;'><code>tcpdump_cookies_off.txt</code> isn't found in your home directory on <code>client</code>. Did you mistype the name of the file?</span>"))
            step4Complete = False

    else:
        output4.clear_output()
        with output4:
            display(HTML("<span style='color: red;'>There was an error checking this step. Please contact your professor/TA.</span>"))
            step4Complete = False

def check_step_4(b):
    if (warn_student(labname)):
        output4.clear_output()
        with output4:
            display(HTML("<span style='color: red;'><strong>WARNING:</strong> You have an autosaved lab that you have not yet loaded. If you would like to load your progress, click \"Load Lab\" at the top of the notebook. Otherwise, clicking on this button again will assume you're restarting the lab!</span>"))
    else:
        step_4()

        # Auto-save.
        if (not runAllSteps):
            trigger_save(labname, "4", result.returncode)

# Creating the button.
button = widgets.Button(description="Check File")

# Creating an output area.
output4 = widgets.Output()

# Run the command on click.
button.on_click(check_step_4)

# Display the output.
display(button, output4)

Button(description='Check File', style=ButtonStyle())

Output()

### Step 5: Collecting Statistics (Part 2)

<strong>Your task:</strong> Now, turn on the SYN cookies and repeat the steps above. Refer to Step 2 if you forgot how to enable SYN cookies.

After completing the steps, navigate to your home (`/home/USERNAME_GOES_HERE`) directory on `client`, then save the file as `tcpdump_cookies_on.txt`.

The notebook will check to see if the file exists. Once again, the notebooks isn't checking for validity. Ensure you saved the file correctly and captured the correct output.

In [7]:
# Click the button below to check your work.
step5Complete = False

# Function to check the permissions.
def step_5():
    # Important variables that must be accessed outside of this function.
    global step5Complete, result

    with output5:
        output5.clear_output()
        display(HTML("<span><img width='14px' height='14px' style='margin-left: 5px;' src='resources/loading.gif'></span>"))
    
    # This subprocess statement is a little different. Need to initiate environment variables at the same time when running the command.
    result = subprocess.run(['ssh', '-i', '/home/USERNAME_GOES_HERE/.ssh/merge_key', 'USERNAME_GOES_HERE@client', '[ -f ~/tcpdump_cookies_on.txt ] && echo 0 || echo 1'], capture_output=True, text=True)
    
    if (result.stdout == "0\n"):
        output5.clear_output()
        with output5:
            display(HTML("<span style='color: green;'><code>tcpdump_cookies_on.txt</code> has been found, and is saved in your submission.</span>"))
            step5Complete = True

    elif (result.stdout == "1\n"):
        output5.clear_output()
        with output5:
            display(HTML("<span style='color: red;'><code>tcpdump_cookies_on.txt</code> isn't found in your home directory on <code>client</code>. Did you mistype the name of the file?</span>"))
            step5Complete = False

    else:
        output5.clear_output()
        with output5:
            display(HTML("<span style='color: red;'>There was an error checking this step. Please contact your professor/TA.</span>"))
            step5Complete = False

def check_step_5(b):
    if (warn_student(labname)):
        output5.clear_output()
        with output5:
            display(HTML("<span style='color: red;'><strong>WARNING:</strong> You have an autosaved lab that you have not yet loaded. If you would like to load your progress, click \"Load Lab\" at the top of the notebook. Otherwise, clicking on this button again will assume you're restarting the lab!</span>"))
    else:
        step_5()

        # Auto-save.
        if (not runAllSteps):
            trigger_save(labname, "5", result.returncode)

# Creating the button.
button = widgets.Button(description="Check File")

# Creating an output area.
output5 = widgets.Output()

# Run the command on click.
button.on_click(check_step_5)

# Display the output.
display(button, output5)

Button(description='Check File', style=ButtonStyle())

Output()

### Step 6: Generating Graphs

Ensure that the files generated from Steps 4 and 5 exist in your home (`/home/USERNAME_GOES_HERE`) directory on `client`.

With the information that you gathered in the previous two steps, generate a graph where the x-axis is time elapsed in seconds, and the y-axis is the connection duration of each packet. Create your graph with the following information.
- Connection duration is the difference between the time of the first SYN and of the ACK following a FIN-ACK (or between the first SYN and the first RESET) on a connection. Recall what uniquely identifies a TCP connection, i.e. how to detect packets that belong to the same connection?
- If a connection did not end with a FIN or a RST, it was assigned to a duration of 200s.
- Indicate on the graphs using vertical lines or arrows the start and the end of the attack.

<strong>Your task:</strong> Draw your graphs with the information provided above. Once you have generated the two graphs, use them in the next step.

This step does not have an automatic grade button.

### Extra Credit Opportunity:

There are two extra credit questions.

1. Remove spoofing from the attack. Repeat the exercise without SYN cookies and observe and explain the effect. What happens? Can you explain why this happens? For hints, run a `tcpdump` on the `server` node and look for traffic patterns.
2. Can you modify the attack so that it is effective without spoofing and how would you do this?

These questions can be answered in the next step.

### Step 7: Interpreting Your Results

Generate a Word document with the following items (label each section):
- Explanation how the TCP SYN flood attack works.
- Explanation how SYN cookies work to prevent denial-of-service effect from SYN flood attack.
- Your legitimate client script.
- Your attack command (from Step 3).
- The connection duration graphs you drew in Step 6 (one with SYN cookies, one without SYN cookies). <u>Remember to draw vertical lines indicating the start/end of the attack.</u>
- Explanation what happens in each case. Is the attack effective? How can you tell this from the graphs?
- Answers to extra credit questions, if any.

This step cannot be automatically graded. 

<u>Optional - If you wish to save a copy within your tarball:</u> Upload a file called `USERNAME_GOES_HERE_submission` with a `.docx` or `.pdf` file extension. Click the button below to save your submission into `saves/USERNAME_GOES_HERE_synflood.tar.gz`.

In [8]:
# Click the button below to save your document.
def step_7():
    with output7:
        output7.clear_output()
        display(HTML("<span><img width='14px' height='14px' style='margin-left: 7px;' src='resources/loading.gif'></span>"))
    
    code = -1
    pdf_path = Path("/home/USERNAME_GOES_HERE/USERNAME_GOES_HERE_submission.pdf")
    docx_path = Path("/home/USERNAME_GOES_HERE/USERNAME_GOES_HERE_submission.docx")
    tarball_path = Path("saves/USERNAME_GOES_HERE_synflood.tar.gz")
    files_added = []

    try:
        if pdf_path.exists() or docx_path.exists():
            with tarfile.open(tarball_path, "a:gz") as tar:
                if pdf_path.exists():
                    tar.add(pdf_path)
                    files_added.append(pdf_path.name)
                if docx_path.exists():
                    tar.add(docx_path)
                    files_added.append(docx_path.name)
            code = 0 if files_added else 1
        else:
            code = 1
    except (FileNotFoundError, tarfile.TarError, OSError) as e:
        code = 2

    with output7:
        output7.clear_output()
        if code == 0:
            display(HTML(f"<span style='color: green;'>Your document(s) {', '.join(files_added)} were successfully included in <code>{tarball_path}</code>.</span>"))
        elif code == 1:
            display(HTML("<span style='color: red;'>Neither <code>USERNAME_GOES_HERE_submission.docx</code> nor <code>USERNAME_GOES_HERE_submission.pdf</code> were found. Nothing was added to the tarball.</span>"))
        else:
            display(HTML(f"<span style='color: red;'>There was an error saving to your tarball. The file <code>{tarball_path}</code> may not exist or is inaccessible. Have you completed any of the previous steps?</span>"))

def check_step_7(b):
    step_7()

# Creating the button.
button = widgets.Button(description="Add To Submission")

# Creating an output area.
output7 = widgets.Output()

# Run the command on click.
button.on_click(check_step_7)

# Display the output.
display(button, output7)

Button(description='Add To Submission', style=ButtonStyle())

Output()

## <strong>Grading</strong>

To check your overall work for this lab, click on the button below.

The notebook automatically saves your work under `saves/USERNAME_GOES_HERE_synflood.tar.gz`. You may be asked to submit this file, or to just submit the work from Step 7. 

<u>Read your assignment's rubric for submission instructions.</u>

In [9]:
# Click the button below to check your overall grade.
steps_to_check = [step_1, step_2, step_3, step_4, step_5]   

# Function to calculate grade after refreshing the cell
def calculate_grade(b):
    # To not auto-save at each step.
    global runAllSteps
    runAllSteps = True

    with gradeOutput:
        gradeOutput.clear_output()
        display(HTML("<span>Testing all steps. Please wait.</span> \
            <span><img width='12px' height='12px' style='margin-left: 3px;' src='resources/loading.gif'></span>"))
    
    # Required for checking the boolean values.
    for func in steps_to_check:
        func()  # Call each function in order.

    # Assuming steps are updated above this cell in some way
    steps = [step1Complete, step2Complete, step3Complete, step4Complete, step5Complete]
    output = ""
    stepsCorrect = 0
    numOfSteps = len(steps)

    for i in range(numOfSteps):
        if steps[i]:
            stepsCorrect += 1
            output += "<div style='color: green;'>Step " + str(i + 1) + " is complete.</div>"
        else:
            output += "<div style='color: red;'>Step " + str(i + 1) + " is incomplete.</div>"

    # Final two steps are not graded.
    output += "<div style='color: orange;'>Step 6 is completed manually, and cannot be auto-graded.</div>"
    output += "<div style='color: orange;'>Step 7 is completed manually, and cannot be auto-graded.</div>"

    # Overall stats.
    output += "<div style='color: black;'>You have " + str(stepsCorrect) + " out of " + str(numOfSteps) + " steps completed.</div>"

    with gradeOutput:
        gradeOutput.clear_output()
        display(HTML(output))

    # Makes auto-saving work again.
    runAllSteps = False
    
# Create a button to refresh the cell and another to calculate grade.
grade_button = widgets.Button(description="Calculate Grade")

# Link buttons to functions.
grade_button.on_click(calculate_grade)

# Output area.
gradeOutput = widgets.Output()

# Display the buttons and output.
display(grade_button, gradeOutput)

Button(description='Calculate Grade', style=ButtonStyle())

Output()

### Stopping the Lab

Once you are done with the lab, click on the "Stop Lab" button below. <strong>This will delete your materialization, which will delete all of the lab's resources.</strong> Your progress is saved automatically in ```saves/``` within the sidebar of your XDC. You may load this lab in the future by clicking "Load Lab" at the top.

In [10]:
# Click the button below to stop the experiment.
def stoplab(button):
    stop_lab(labname, confirm, stop_output)

# Creating the button.
stopButton = widgets.Button(description="Stop Lab")

# Create a confirmation check.
confirm = widgets.Checkbox(
    value=False,
    description='Confirm',
    disabled=False,
    indent=False
)

# Creating an output area.
stop_output = widgets.Output()

# Run the command on click.
stopButton.on_click(stoplab)

# Display the output.
display(confirm, stopButton, stop_output)

Checkbox(value=False, description='Confirm', indent=False)

Button(description='Stop Lab', style=ButtonStyle())

Output()